# PINN-MPC v5 — MPPI + Physics-Consistent PINN + Optuna

## Key changes from v4
```
PINN:
  - Tanh -> SiLU  (less gradient saturation)
  - 3 layers -> 4 layers, hidden 128
  - Physics loss: structured kinematics constraint
      l1 = (dh_pred    - V*sin(theta)*dt)^2
      l2 = (dtheta_pred - q*dt)^2
      l3 = smoothness: mean(dq_pred^2)  <- pitch rate explosion prevention
  - Consistent input/output normalisation in rollout

MPC:
  - CEM -> MPPI  (soft weighting, smoother control)
  - horizon 10/50 -> 20/50
  - Cost: altitude error + pitch constraint penalty

Tuning:
  - Optuna PINN HPO  (target rollout RMSE < 3.28 ft = 1 m)
  - Optuna PID tuning
  - Iterative data collection if RMSE target not met

Safety:
  - MPC early termination on divergence / crash / uncontrolled pitch
  - All plot labels in English
```

## 0. Install & Import

In [1]:
!pip install jsbsim optuna -q
print('Install complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 40.5 MB/s eta 0:00:00
Install complete


In [2]:
import jsbsim
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import optuna
import time, os, warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/PINN_MPC_v5'
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
np.random.seed(42); torch.manual_seed(42)

# ── Constants ─────────────────────────────────────────
INIT_ALT        = 3000    # ft
TARGET_ALT      = 5000    # ft
THROTTLE        = 0.85
DT              = 0.02    # s
MAX_STEPS       = 3000    # 60 s
RMSE_TARGET_FT  = 3.28    # 1 m
MAX_DATA_ROUNDS = 5
DIVERGE_DROP_FT = 200     # early-termination threshold

Mounted at /content/drive
Device : cuda


## 1. JSBSim Helpers

In [3]:
def make_fdm(init_alt=INIT_ALT, init_speed=100, init_pitch=0):
    fdm = jsbsim.FGFDMExec(None, None)
    fdm.set_debug_level(0)
    fdm.load_model('c172p')
    fdm['ic/h-sl-ft']      = init_alt
    fdm['ic/vt-kts']       = init_speed
    fdm['ic/theta-deg']    = init_pitch
    fdm['ic/psi-true-deg'] = 0
    fdm.run_ic()
    fdm['propulsion/engine/set-running'] = 1
    fdm['fcs/throttle-cmd-norm']         = THROTTLE
    return fdm

def read_state(fdm):
    """Returns [h(ft), V(fps), theta(rad), q(rad/s)]"""
    return np.array([
        fdm['position/h-sl-ft'],
        fdm['velocities/vt-fps'],
        fdm['attitude/theta-rad'],
        fdm['velocities/q-rad_sec']
    ])

print('JSBSim helpers ready')

JSBSim helpers ready


## 2. Data Collection

In [4]:
def collect_episodes(n_episodes=20, n_steps=1000, verbose=True):
    rows = []
    for ep in range(n_episodes):
        fdm  = make_fdm(
            init_alt   = np.random.uniform(2000, 7000),
            init_speed = np.random.uniform(80, 120),
            init_pitch = np.random.uniform(-8, 12)
        )
        elev   = 0.0
        prev_s = None

        for _ in range(n_steps):
            elev = 0.92 * elev + 0.08 * np.random.uniform(-0.4, 0.4)
            fdm['fcs/elevator-cmd-norm'] = elev
            fdm.run()
            s = np.array([
                fdm['position/h-sl-ft'],
                fdm['velocities/vt-fps'],
                fdm['attitude/theta-rad'],
                fdm['velocities/q-rad_sec'],
                elev
            ])
            if prev_s is not None and abs(s[0]) < 30000 and 50 < s[1] < 1000:
                rows.append({
                    'h': prev_s[0], 'V': prev_s[1],
                    'theta': prev_s[2], 'q': prev_s[3],
                    'elevator': prev_s[4],
                    'dh':     s[0] - prev_s[0],
                    'dV':     s[1] - prev_s[1],
                    'dtheta': s[2] - prev_s[2],
                    'dq':     s[3] - prev_s[3],
                })
            prev_s = s.copy()

        if verbose:
            print(f'  ep {ep+1:2d}/{n_episodes} | rows: {len(rows):,}')

    return pd.DataFrame(rows)


print('Collecting initial dataset (20 episodes)...')
df_all = collect_episodes(n_episodes=20)
print(f'Total: {len(df_all):,} samples')



     JSBSim Flight Dynamics Model v1.3.0 Apr  9 2026 10:00:08
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

  ep  1/20 | rows: 999
  ep  2/20 | rows: 1,998
  ep  3/20 | rows: 2,997
  ep  4/20 | rows: 3,996
  ep  5/20 | rows: 4,995
  ep  6/20 | rows: 5,994
  ep  7/20 | rows: 6,993
  ep  8/20 | rows: 7,992
  ep  9/20 | rows: 8,991
  ep 10/20 | rows: 9,990
  ep 11/20 | rows: 10,989
  ep 12/20 | rows: 11,988
  ep 13/20 | rows: 12,987
  ep 14/20 | rows: 13,986
  ep 15/20 | rows: 14,985
  ep 16/20 | rows: 15,984
  ep 17/20 | rows: 16,983
  ep 18/20 | rows: 17,982
  ep 19/20 | rows: 18,981
  ep 20/20 | rows: 19,980
Total: 19,980 samples


## 3. Physics-Consistent PINN

### Key improvements
```
① Activation: Tanh -> SiLU  (no gradient saturation)
② Depth: 3 layers -> 4 layers
③ Physics loss — structured kinematics:
     l1 = (dh_pred    - V*sin(theta)*dt)^2   <- altitude kinematics
     l2 = (dtheta_pred - q*dt)^2             <- pitch kinematics
     l3 = mean(dq_pred^2)                    <- pitch rate smoothness
④ Rollout: fully consistent normalisation
     delta = d_norm * Y_std  (do NOT add Y_mean)
```

In [5]:
class PhysicsPINN(nn.Module):
    """
    Physics-Consistent PINN
    Input:  [h, V, theta, q, elevator]  (5, normalised)
    Output: [dh, dV, dtheta, dq]        (4, normalised deltas)

    Physics loss (in physical units, denormalised):
      l1 = (dh_pred    - V*sin(theta)*dt)^2
      l2 = (dtheta_pred - q*dt)^2
      l3 = mean(dq_pred^2)   <- smoothness regulariser
    """
    def __init__(self, hidden_dim=128, n_layers=4,
                 Xm=None, Xs=None, Ym=None, Ys=None):
        super().__init__()
        # Store normalisation stats for physics loss denormalisation
        self.register_buffer('Xm', Xm if Xm is not None else torch.zeros(5))
        self.register_buffer('Xs', Xs if Xs is not None else torch.ones(5))
        self.register_buffer('Ym', Ym if Ym is not None else torch.zeros(4))
        self.register_buffer('Ys', Ys if Ys is not None else torch.ones(4))

        # ✅ SiLU activation — less gradient saturation than Tanh
        layers = [nn.Linear(5, hidden_dim), nn.SiLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.SiLU()]
        layers.append(nn.Linear(hidden_dim, 4))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def physics_loss(self, x_norm, lam_smooth=0.01, dt=DT):
        """
        x_norm: normalised input batch
        Computes physics loss in physical (denormalised) units.
        """
        d_norm = self.net(x_norm)   # (B, 4) normalised deltas

        # Denormalise inputs
        x_phys = x_norm * self.Xs + self.Xm   # (B, 5)
        V      = x_phys[:, 1]
        theta  = x_phys[:, 2]
        q      = x_phys[:, 3]

        # Denormalise outputs (only std, NOT mean — delta quantity)
        d_phys = d_norm * self.Ys              # (B, 4)
        dh_pred     = d_phys[:, 0]
        dtheta_pred = d_phys[:, 2]
        dq_pred     = d_phys[:, 3]

        # ── Structured kinematics constraints ──────────
        # l1: altitude kinematics  dh = V*sin(theta)*dt
        dh_phys = V * torch.sin(theta) * dt
        l1 = torch.mean((dh_pred - dh_phys) ** 2)

        # l2: pitch kinematics  dtheta = q*dt
        dtheta_phys = q * dt
        l2 = torch.mean((dtheta_pred - dtheta_phys) ** 2)

        # l3: pitch rate smoothness (prevent dq explosion)
        l3 = lam_smooth * torch.mean(dq_pred ** 2)

        return l1 + l2 + l3


def build_tensors(df):
    feat = ['h', 'V', 'theta', 'q', 'elevator']
    tgt  = ['dh', 'dV', 'dtheta', 'dq']
    X = torch.FloatTensor(df[feat].values)
    Y = torch.FloatTensor(df[tgt].values)

    Xm, Xs = X.mean(0), X.std(0) + 1e-8
    Ym, Ys = Y.mean(0), Y.std(0) + 1e-8
    Xn = (X - Xm) / Xs
    Yn = (Y - Ym) / Ys

    n   = len(Xn)
    idx = torch.randperm(n)
    sp  = int(n * 0.8)
    return (Xn[idx[:sp]].to(device), Yn[idx[:sp]].to(device),
            Xn[idx[sp:]].to(device), Yn[idx[sp:]].to(device),
            Xm, Xs, Ym, Ys)


print('PhysicsPINN class ready')

PhysicsPINN class ready


## 4. Rollout — Consistent Normalisation

In [6]:
def make_rollout_fn(model, Xm, Xs, Ym, Ys):
    """
    Returns a rollout function bound to the given model.

    Normalisation contract:
      x_norm  = (x - X_mean) / X_std
      d_norm  = model(x_norm)
      delta   = d_norm * Y_std          <- ✅ do NOT add Y_mean
      state  += delta
    """
    xm = Xm.numpy(); xs = Xs.numpy()
    ys = Ys.numpy()

    def rollout(state, elevator_seq):
        model.eval()
        states  = [state.copy()]
        current = state.copy()
        for elev in elevator_seq:
            x  = np.array([*current, elev])
            xn = (x - xm) / xs
            xt = torch.FloatTensor(xn).unsqueeze(0).to(device)
            with torch.no_grad():
                dn = model(xt).cpu().numpy()[0]
            delta   = dn * ys   # ✅ only Y_std, not Y_mean
            current = current + delta
            states.append(current.copy())
        return np.array(states)

    return rollout


def eval_rollout_rmse(model, Xm, Xs, Ym, Ys,
                      n_eval=5, n_steps=150):
    """Mean altitude RMSE over n_eval JSBSim episodes."""
    rollout = make_rollout_fn(model, Xm, Xs, Ym, Ys)
    rmses   = []
    for _ in range(n_eval):
        fdm    = make_fdm(
            init_alt   = np.random.uniform(2500, 5500),
            init_speed = np.random.uniform(90, 110)
        )
        elevs  = np.sin(np.linspace(0, 3*np.pi, n_steps)) * 0.15
        actual = []; init_s = None
        for i, e in enumerate(elevs):
            fdm['fcs/elevator-cmd-norm'] = e
            fdm.run()
            s = read_state(fdm)
            actual.append(s)
            if i == 0: init_s = s.copy()
        actual    = np.array(actual)
        predicted = rollout(init_s, elevs)
        rmses.append(np.sqrt(np.mean(
            (actual[:, 0] - predicted[1:, 0])**2
        )))
    return float(np.mean(rmses))


print('Rollout functions ready')

Rollout functions ready


## 5. Optuna PINN HPO + Iterative Data Collection

```
Loop (max 5 rounds):
  1. Optuna 30 trials -> find best PINN hyperparams
  2. Evaluate rollout RMSE
  3. RMSE < 3.28 ft?  -> done
                No    -> add more data, repeat
```

In [7]:
def train_one_trial(trial, Xtr, Ytr, Xvl, Yvl, Xm, Xs, Ym, Ys):
    hidden_dim  = trial.suggest_categorical('hidden_dim', [64, 128, 256])
    n_layers    = trial.suggest_int('n_layers', 3, 5)
    lr          = trial.suggest_float('lr', 5e-5, 3e-3, log=True)
    batch_size  = trial.suggest_categorical('batch_size', [256, 512, 1024])
    lam_physics = trial.suggest_float('lam_physics', 0.05, 2.0, log=True)
    lam_smooth  = trial.suggest_float('lam_smooth',  0.001, 0.1, log=True)
    epochs      = trial.suggest_categorical('epochs', [600, 800, 1000])

    model = PhysicsPINN(hidden_dim, n_layers,
                        Xm.to(device), Xs.to(device),
                        Ym.to(device), Ys.to(device)).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    crit  = nn.MSELoss()

    best_val = np.inf; best_sd = None

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        tl, nb = 0, 0
        for i in range(0, len(Xtr), batch_size):
            xb = Xtr[perm[i:i+batch_size]]
            yb = Ytr[perm[i:i+batch_size]]
            ld   = crit(model(xb), yb)
            lp   = model.physics_loss(xb, lam_smooth)
            loss = ld + lam_physics * lp
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += ld.item(); nb += 1
        sch.step()

        model.eval()
        with torch.no_grad():
            vl = crit(model(Xvl), Yvl).item()
        if vl < best_val:
            best_val = vl
            best_sd  = {k: v.clone().cpu()
                        for k, v in model.state_dict().items()}

        trial.report(vl, ep)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    model.load_state_dict({k: v.to(device) for k, v in best_sd.items()})
    return model, best_val


print('Optuna trainer ready')

Optuna trainer ready


In [8]:
# ── Main HPO loop ─────────────────────────────────────────────
best_pinn   = None
best_stats  = None
best_rmse   = np.inf
best_params = None
round_log   = []

for rnd in range(MAX_DATA_ROUNDS):
    n_samp = len(df_all)
    print(f'\n=== Round {rnd+1}/{MAX_DATA_ROUNDS} | '
          f'dataset: {n_samp:,} samples ===')

    Xtr, Ytr, Xvl, Yvl, Xm, Xs, Ym, Ys = build_tensors(df_all)

    study = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=5, n_warmup_steps=150
        )
    )

    def objective(trial):
        model, _ = train_one_trial(
            trial, Xtr, Ytr, Xvl, Yvl, Xm, Xs, Ym, Ys
        )
        rmse = eval_rollout_rmse(model, Xm, Xs, Ym, Ys)
        trial.set_user_attr('rmse', rmse)
        trial.set_user_attr('model_sd',
            {k: v.cpu() for k, v in model.state_dict().items()})
        trial.set_user_attr('hidden_dim',
            trial.params.get('hidden_dim', 128))
        trial.set_user_attr('n_layers',
            trial.params.get('n_layers', 4))
        return rmse

    print('  Running Optuna (30 trials)...')
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    bt       = study.best_trial
    rnd_rmse = bt.user_attrs['rmse']
    print(f'  Best RMSE : {rnd_rmse:.3f} ft  '
          f'(target < {RMSE_TARGET_FT:.2f} ft)')
    print(f'  Params    : {bt.params}')

    round_log.append({
        'round':   rnd + 1,
        'n_data':  n_samp,
        'rmse_ft': round(rnd_rmse, 3),
        'params':  bt.params
    })

    if rnd_rmse < best_rmse:
        best_rmse   = rnd_rmse
        best_params = bt.params
        best_pinn   = PhysicsPINN(
            bt.user_attrs['hidden_dim'],
            bt.user_attrs['n_layers'],
            Xm.to(device), Xs.to(device),
            Ym.to(device), Ys.to(device)
        ).to(device)
        best_pinn.load_state_dict(
            {k: v.to(device)
             for k, v in bt.user_attrs['model_sd'].items()}
        )
        best_stats = (Xm, Xs, Ym, Ys)
        torch.save({
            'model_state': best_pinn.state_dict(),
            'Xm': Xm, 'Xs': Xs, 'Ym': Ym, 'Ys': Ys,
            'params': bt.params,
            'rmse_ft': rnd_rmse
        }, f'{SAVE_DIR}/pinn_best.pt')

    if best_rmse <= RMSE_TARGET_FT:
        print(f'\n[PASS] RMSE = {best_rmse:.3f} ft <= '
              f'{RMSE_TARGET_FT:.2f} ft  -> target achieved')
        break

    if rnd < MAX_DATA_ROUNDS - 1:
        add_eps = 10 * (rnd + 2)
        print(f'\n  [RETRY] Adding {add_eps} episodes...')
        df_extra = collect_episodes(n_episodes=add_eps, verbose=False)
        df_all   = pd.concat([df_all, df_extra], ignore_index=True)
        print(f'  Dataset: {len(df_all):,}')
    else:
        print(f'\n[WARN] Max rounds reached.  Best RMSE: {best_rmse:.3f} ft')

print('\n=== HPO Summary ===')
for r in round_log:
    flag = '<< PASS' if r['rmse_ft'] <= RMSE_TARGET_FT else ''
    print(f"  Round {r['round']} | n={r['n_data']:,} | "
          f"RMSE={r['rmse_ft']:.3f}ft {flag}")
print(f'\nFinal best RMSE : {best_rmse:.3f} ft')
print(f'Best params     : {best_params}')


=== Round 1/5 | dataset: 19,980 samples ===
  Running Optuna (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

  Best RMSE : 10.330 ft  (target < 3.28 ft)
  Params    : {'hidden_dim': 128, 'n_layers': 5, 'lr': 0.0012619667520220362, 'batch_size': 512, 'lam_physics': 0.10268730652684648, 'lam_smooth': 0.048737593198974916, 'epochs': 800}

  [RETRY] Adding 20 episodes...
  Dataset: 39,960

=== Round 2/5 | dataset: 39,960 samples ===
  Running Optuna (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

[W 2026-04-28 10:14:25,428] Trial 11 failed with parameters: {'hidden_dim': 128, 'n_layers': 4, 'lr': 0.0012752261257015817, 'batch_size': 1024, 'lam_physics': 0.17279029687665026, 'lam_smooth': 0.02272250860425673, 'epochs': 1000} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1572/4190769820.py", line 23, in objective
    model, _ = train_one_trial(
               ^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1572/3409709672.py", line 29, in train_one_trial
    opt.zero_grad(); loss.backward()
                     ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/__init__.py", line 364, in backward
    _engine_run_bac

KeyboardInterrupt: 

## 6. Rollout Validation Plot

In [ ]:
Xm, Xs, Ym, Ys = best_stats
rollout = make_rollout_fn(best_pinn, Xm, Xs, Ym, Ys)


def validate_rollout_plot(n_steps=200):
    fdm    = make_fdm()
    elevs  = np.sin(np.linspace(0, 4*np.pi, n_steps)) * 0.15
    actual = []; init_s = None

    for i, e in enumerate(elevs):
        fdm['fcs/elevator-cmd-norm'] = e
        fdm.run()
        s = read_state(fdm)
        actual.append(s)
        if i == 0: init_s = s.copy()

    actual    = np.array(actual)
    predicted = rollout(init_s, elevs)

    labels = ['Altitude (ft)', 'Velocity (fps)',
              'Pitch (rad)', 'Pitch Rate (rad/s)']
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    rmses = []
    for i, (ax, lb) in enumerate(zip(axes.flat, labels)):
        ax.plot(actual[:, i],     color='steelblue',
                linewidth=1.8, label='JSBSim')
        ax.plot(predicted[1:, i], color='tomato',
                linewidth=1.8, linestyle='--', label='PINN')
        r = np.sqrt(np.mean((actual[:, i] - predicted[1:, i])**2))
        rmses.append(r)
        ax.set_title(f'{lb}  RMSE={r:.4f}', fontsize=10)
        ax.set_xlabel('Step'); ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    status = 'PASS' if rmses[0] <= RMSE_TARGET_FT else 'WARN'
    plt.suptitle(
        f'PINN Rollout Validation  [{status}]  '
        f'Alt RMSE={rmses[0]:.3f}ft  target<{RMSE_TARGET_FT:.2f}ft',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/rollout_validation.png', dpi=150)
    plt.show()
    print(f'Altitude RMSE : {rmses[0]:.4f} ft  |  Target: {RMSE_TARGET_FT:.2f} ft')


validate_rollout_plot()

## 7. Optuna PID Tuning

In [ ]:
def eval_pid(kp_alt, kp, ki, kd, n_steps=1500):
    fdm = make_fdm()
    integral = 0.0; prev_err = 0.0
    ILIM = 5.0; alts = []

    for _ in range(n_steps):
        h, V, theta, q = read_state(fdm)
        alts.append(h)

        if h < 500 or h > 15000 or abs(np.degrees(theta)) > 60:
            return 1e6

        theta_cmd = np.clip((TARGET_ALT - h) * kp_alt, -0.20, 0.20)
        err       = theta_cmd - theta
        integral  = np.clip(integral + err*DT, -ILIM, ILIM)
        deriv     = (err - prev_err) / DT
        prev_err  = err

        elev = np.clip(kp*err + ki*integral + kd*deriv, -1.0, 1.0)
        fdm['fcs/elevator-cmd-norm'] = elev
        fdm.run()

    return float(np.sqrt(np.mean((np.array(alts) - TARGET_ALT)**2)))


def pid_objective(trial):
    return eval_pid(
        trial.suggest_float('kp_alt', 0.0001, 0.002,  log=True),
        trial.suggest_float('kp',     0.1,    2.0),
        trial.suggest_float('ki',     0.001,  0.2,    log=True),
        trial.suggest_float('kd',     0.01,   1.0,    log=True)
    )


print('Running Optuna PID tuning (50 trials)...')
pid_study = optuna.create_study(direction='minimize')
pid_study.optimize(pid_objective, n_trials=50, show_progress_bar=True)

bp = pid_study.best_params
print(f'\nBest PID params:')
for k, v in bp.items():
    print(f'  {k:8s} = {v:.6f}')
print(f'  RMSE     = {pid_study.best_value:.2f} ft')

## 8. Controllers

### MPPI vs CEM
```
CEM  (v4): top-k selection  -> discontinuous update, local minima
MPPI (v5): soft weighting   -> smooth, gradient-like update

MPPI weight:  w_i = exp(-cost_i / lambda)
New mean:     u   = sum(w_i * noise_i) / sum(w_i)

Effect:
  - All samples contribute (weighted by quality)
  - Better samples get exponentially more weight
  - Control output is smooth
```

In [ ]:
# ── Cost function ──────────────────────────────────────────────
def mpc_cost(predicted, target_alt, elev_seq):
    """
    Cost components:
      c_alt      : altitude tracking (normalised)
      c_terminal : final step altitude penalty
      c_safety   : pitch angle limit violation (>= 0.3 rad = 17 deg)
      c_comfort  : pitch rate minimisation
      c_smooth   : elevator change rate
    """
    h     = predicted[:, 0]
    theta = predicted[:, 2]
    q     = predicted[:, 3]

    c_alt      = 2.0 * np.mean((h - target_alt)**2) / (target_alt**2)
    c_terminal = 3.0 * ((h[-1] - target_alt)**2)    / (target_alt**2)
    c_safety   = 3.0 * np.sum(np.maximum(0, np.abs(theta) - 0.30)**2)
    c_comfort  = 0.001 * np.mean(q**2)
    c_smooth   = 0.01  * (np.mean(np.diff(elev_seq)**2)
                          if len(elev_seq) > 1 else 0)
    return c_alt + c_terminal + c_safety + c_comfort + c_smooth


# ── Controller 1: Tuned PID ────────────────────────────────────
class TunedPID:
    def __init__(self, params):
        self.kp_alt = params['kp_alt']
        self.kp = params['kp']
        self.ki = params['ki']
        self.kd = params['kd']
        self.ILIM = 5.0
        self.integral = 0.0; self.prev_err = 0.0

    def update(self, state, target_alt, dt=DT):
        h, V, theta, q = state
        theta_cmd = np.clip((target_alt - h) * self.kp_alt, -0.20, 0.20)
        err       = theta_cmd - theta
        self.integral = np.clip(
            self.integral + err*dt, -self.ILIM, self.ILIM
        )
        deriv         = (err - self.prev_err) / dt
        self.prev_err = err
        elev = self.kp*err + self.ki*self.integral + self.kd*deriv
        return np.clip(elev, -1.0, 1.0), theta_cmd

    def reset(self):
        self.integral = 0.0; self.prev_err = 0.0


# ── Controller 2 & 3: PINN-MPC MPPI ───────────────────────────
class PINN_MPC_MPPI:
    """
    Model Predictive Path Integral (MPPI) controller.

    Each step:
      1. Sample N noise sequences  eps ~ N(0, sigma)
      2. Apply to current mean:    u_i = U + eps_i
      3. Rollout each with PINN -> cost_i
      4. Compute soft weights:     w_i = exp(-cost_i / lam)
      5. Update mean:              U  += sum(w_i * eps_i) / sum(w_i)
      6. Execute U[0], shift U left (warm start)

    vs CEM:
      CEM uses hard top-k selection  -> discontinuous, local minima
      MPPI uses soft weighting       -> smooth, gradient-like update
    """
    def __init__(self, horizon=20, n_samples=64,
                 sigma=0.20, lam=0.05):
        self.horizon   = horizon
        self.n_samples = n_samples
        self.sigma     = sigma
        self.lam       = lam          # temperature: lower = more greedy
        self.U         = np.zeros(horizon)   # nominal sequence

    def update(self, state, target_alt):
        # Sample noise
        eps = np.random.normal(0, self.sigma,
                               size=(self.n_samples, self.horizon))

        # Perturbed sequences
        seqs = np.clip(self.U + eps, -0.35, 0.35)   # (N, H)

        # Rollout + cost
        costs = np.array([
            mpc_cost(rollout(state, seq), target_alt, seq)
            for seq in seqs
        ])

        # ── MPPI soft weighting ──────────────────────
        beta    = costs.min()                        # numerical stability
        weights = np.exp(-(costs - beta) / self.lam) # (N,)
        weights = weights / (weights.sum() + 1e-8)   # normalise

        # Weighted update of nominal sequence
        self.U = self.U + np.sum(
            weights[:, None] * eps, axis=0
        )
        self.U = np.clip(self.U, -0.35, 0.35)

        best_cost = float(np.sum(weights * costs))

        # Execute first action, shift sequence (warm start)
        action = self.U[0]
        self.U = np.append(self.U[1:], self.U[-1])

        return action, best_cost

    def reset(self):
        self.U = np.zeros(self.horizon)


print('Controllers ready')
print('  TunedPID      : Optuna-tuned gains')
print('  PINN-MPC MPPI horizon=20  (0.4s lookahead)')
print('  PINN-MPC MPPI horizon=50  (1.0s lookahead)')

## 9. Simulation Runner with Early Termination

In [ ]:
def run_sim(controller, name,
            target_alt=TARGET_ALT, max_steps=MAX_STEPS):
    """
    Early termination conditions:
      DIVERGE     : alt drops > DIVERGE_DROP_FT below INIT_ALT
      CRASH       : alt < 500 ft
      UNCONTROLLED: |pitch| > 60 deg
    """
    fdm = make_fdm()
    controller.reset()

    log = {k: [] for k in [
        'time', 'altitude', 'velocity',
        'pitch', 'pitch_rate',
        'elevator', 'theta_cmd',
        'cost', 'comp_time'
    ]}

    print(f'\n[{name}]  target={target_alt}ft')
    reached = False; abort_reason = None

    for step in range(max_steps):
        state        = read_state(fdm)
        h, V, theta, q = state
        t0           = time.perf_counter()

        # ── Early termination ──────────────────────────
        if h < 500:
            abort_reason = f'CRASH  alt={h:.0f}ft'
            break
        if h < INIT_ALT - DIVERGE_DROP_FT:
            abort_reason = (
                f'DIVERGE  alt={h:.0f}ft  '
                f'(>{DIVERGE_DROP_FT}ft below start)'
            )
            break
        if abs(np.degrees(theta)) > 60:
            abort_reason = f'UNCONTROLLED  pitch={np.degrees(theta):.1f}deg'
            break

        # ── Control update ─────────────────────────────
        if isinstance(controller, TunedPID):
            elev, theta_cmd = controller.update(state, target_alt)
            cost = abs(h - target_alt)
        else:
            elev, cost  = controller.update(state, target_alt)
            theta_cmd   = np.nan

        comp_ms = (time.perf_counter() - t0) * 1000

        fdm['fcs/elevator-cmd-norm'] = elev
        fdm.run()

        log['time'].append(step * DT)
        log['altitude'].append(h)
        log['velocity'].append(V)
        log['pitch'].append(np.degrees(theta))
        log['pitch_rate'].append(np.degrees(q))
        log['elevator'].append(elev)
        log['theta_cmd'].append(
            np.degrees(theta_cmd) if not np.isnan(theta_cmd) else np.nan
        )
        log['cost'].append(cost)
        log['comp_time'].append(comp_ms)

        if step % 500 == 0:
            print(f'  t={step*DT:5.1f}s | '
                  f'alt={h:7.1f}ft | '
                  f'pitch={np.degrees(theta):6.1f}deg | '
                  f'comp={comp_ms:.1f}ms')

        if abs(h - target_alt) < 100 and not reached:
            print(f'  [REACHED] t={step*DT:.1f}s | alt={h:.1f}ft')
            reached = True
            break

    if abort_reason:
        print(f'  [ABORT]   t={step*DT:.1f}s | {abort_reason}')
    elif not reached:
        print(f'  [TIMEOUT] final alt={h:.1f}ft')

    log['abort_reason'] = abort_reason
    return log


pid    = TunedPID(bp)
mppi20 = PINN_MPC_MPPI(horizon=20, n_samples=64, sigma=0.20, lam=0.05)
mppi50 = PINN_MPC_MPPI(horizon=50, n_samples=64, sigma=0.20, lam=0.05)

log_pid    = run_sim(pid,    'Tuned PID')
log_mppi20 = run_sim(mppi20, 'PINN-MPC MPPI horizon=20')
log_mppi50 = run_sim(mppi50, 'PINN-MPC MPPI horizon=50')

print('\nAll simulations done')

## 10. Metrics

In [ ]:
def compute_metrics(log, target_alt, name):
    alt   = np.array(log['altitude'])
    pitch = np.array(log['pitch'])
    q     = np.array(log['pitch_rate'])
    comp  = np.array(log['comp_time'])
    t     = np.array(log['time'])

    reached = np.where(np.abs(alt - target_alt) < 100)[0]
    reach_t = round(float(t[reached[0]]), 1) if len(reached) > 0 else 'N/A'
    ss_idx  = int(len(alt) * 0.9)

    return {
        'Controller':      name,
        'Reach Time (s)':  reach_t,
        'SS Error (ft)':   round(float(np.mean(np.abs(alt[ss_idx:] - target_alt))), 1),
        'Alt RMSE (ft)':   round(float(np.sqrt(np.mean((alt - target_alt)**2))), 1),
        'Max Pitch (deg)': round(float(np.max(np.abs(pitch))), 1),
        'Pitch RMS (d/s)': round(float(np.sqrt(np.mean(q**2))), 3),
        'Constr Viol':     int(np.sum(np.abs(pitch) > 17)),  # 0.3 rad = 17 deg
        'Mean Comp (ms)':  round(float(np.mean(comp)), 2),
        'Abort':           log['abort_reason'] or 'None',
    }


m_pid    = compute_metrics(log_pid,    TARGET_ALT, 'Tuned PID')
m_mppi20 = compute_metrics(log_mppi20, TARGET_ALT, 'MPPI horizon=20')
m_mppi50 = compute_metrics(log_mppi50, TARGET_ALT, 'MPPI horizon=50')

mdf = pd.DataFrame([m_pid, m_mppi20, m_mppi50]).set_index('Controller')
print('\n=== Performance Comparison ===')
print(mdf.to_string())
mdf.to_csv(f'{SAVE_DIR}/metrics_v5.csv')

## 11. Visualisation

In [ ]:
def plot_comparison(log_pid, log_mppi20, log_mppi50, target_alt):
    C    = {'pid': '#2196F3', 'h20': '#FF9800', 'h50': '#4CAF50'}
    L    = {'pid': 'Tuned PID',
            'h20': 'PINN-MPC MPPI h=20',
            'h50': 'PINN-MPC MPPI h=50'}
    logs = {'pid': log_pid, 'h20': log_mppi20, 'h50': log_mppi50}

    fig = plt.figure(figsize=(16, 15))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.35)

    # 1. Altitude
    ax1  = fig.add_subplot(gs[0, :])
    max_t = max(max(log['time']) for log in logs.values())
    for k, log in logs.items():
        lbl = L[k]
        if log['abort_reason']:
            lbl += f'  [ABORT]'
        ax1.plot(log['time'], log['altitude'],
                 color=C[k], label=lbl, linewidth=1.8)
    ax1.axhline(target_alt, color='red', linestyle='--',
                linewidth=1.5, label=f'Target {target_alt}ft')
    ax1.fill_between([0, max_t],
                     target_alt-100, target_alt+100,
                     alpha=0.1, color='red', label='Tol +/-100ft')
    ax1.axhline(INIT_ALT - DIVERGE_DROP_FT, color='black',
                linestyle=':', linewidth=1.2,
                label=f'Abort threshold ({INIT_ALT-DIVERGE_DROP_FT}ft)')
    ax1.set_ylabel('Altitude (ft)', fontsize=11)
    ax1.set_xlabel('Time (s)')
    ax1.set_title('Altitude Tracking', fontsize=13, fontweight='bold')
    ax1.legend(loc='lower right', fontsize=8)
    ax1.grid(True, alpha=0.3)

    # 2. Pitch angle
    ax2 = fig.add_subplot(gs[1, 0])
    for k, log in logs.items():
        ax2.plot(log['time'], log['pitch'],
                 color=C[k], label=L[k], linewidth=1.5)
    ax2.axhline( 17, color='gray', linestyle=':', alpha=0.8)
    ax2.axhline(-17, color='gray', linestyle=':', alpha=0.8,
                label='+/-17 deg limit')
    ax2.set_ylabel('Pitch (deg)'); ax2.set_xlabel('Time (s)')
    ax2.set_title('Pitch Angle')
    ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)

    # 3. Pitch rate
    ax3 = fig.add_subplot(gs[1, 1])
    for k, log in logs.items():
        ax3.plot(log['time'], log['pitch_rate'],
                 color=C[k], label=L[k], linewidth=1.5)
    ax3.set_ylabel('Pitch Rate (deg/s)'); ax3.set_xlabel('Time (s)')
    ax3.set_title('Pitch Rate  (lower = more comfortable)')
    ax3.legend(fontsize=7); ax3.grid(True, alpha=0.3)

    # 4. Elevator
    ax4 = fig.add_subplot(gs[1, 2])
    for k, log in logs.items():
        ax4.plot(log['time'], log['elevator'],
                 color=C[k], label=L[k], linewidth=1.5)
    ax4.axhline(0, color='gray', linestyle='-', alpha=0.4)
    ax4.set_ylabel('Elevator'); ax4.set_xlabel('Time (s)')
    ax4.set_title('Elevator Input')
    ax4.legend(fontsize=7); ax4.grid(True, alpha=0.3)

    # 5. Computation time
    ax5 = fig.add_subplot(gs[2, 0])
    for k, log in logs.items():
        ax5.plot(log['time'], log['comp_time'],
                 color=C[k], label=L[k], linewidth=1.2, alpha=0.8)
    ax5.set_ylabel('Comp Time (ms)'); ax5.set_xlabel('Time (s)')
    ax5.set_title('Computation Time  (trade-off)')
    ax5.legend(fontsize=7); ax5.grid(True, alpha=0.3)

    # 6. Altitude error
    ax6 = fig.add_subplot(gs[2, 1])
    for k, log in logs.items():
        err = np.abs(np.array(log['altitude']) - target_alt)
        ax6.plot(log['time'], err,
                 color=C[k], label=L[k], linewidth=1.5)
    ax6.axhline(100, color='red', linestyle='--',
                alpha=0.7, label='100ft')
    ax6.set_ylabel('Alt Error (ft)'); ax6.set_xlabel('Time (s)')
    ax6.set_title('Absolute Altitude Error')
    ax6.legend(fontsize=7); ax6.grid(True, alpha=0.3)

    # 7. Radar chart
    ax7  = fig.add_subplot(gs[2, 2], polar=True)
    cats = ['Tracking\nAcc', 'Comfort', 'Safety',
            'Response\nSpeed', 'Comp\nEfficiency']
    N    = len(cats)
    ang  = [n/N*2*np.pi for n in range(N)]; ang += ang[:1]
    mlist = [m_pid, m_mppi20, m_mppi50]
    rkeys = ['Alt RMSE (ft)', 'Pitch RMS (d/s)',
             'Constr Viol', 'Reach Time (s)', 'Mean Comp (ms)']

    def sf(v):
        try: return float(v)
        except: return 9999.0

    avals = {rk: [sf(m[rk]) for m in mlist] for rk in rkeys}
    for k, m in zip(['pid', 'h20', 'h50'], mlist):
        scores = []
        for rk in rkeys:
            worst = max(avals[rk]); best = min(avals[rk])
            scores.append(1 - np.clip(
                (sf(m[rk]) - best) / (worst - best + 1e-8), 0, 1
            ))
        scores += scores[:1]
        ax7.plot(ang, scores, color=C[k], linewidth=2, label=L[k])
        ax7.fill(ang, scores, color=C[k], alpha=0.1)

    ax7.set_xticks(ang[:-1])
    ax7.set_xticklabels(cats, fontsize=7)
    ax7.set_ylim(0, 1)
    ax7.set_title('Overall Performance\n(larger = better)',
                  fontsize=9, pad=15)
    ax7.legend(loc='upper right',
               bbox_to_anchor=(1.45, 1.15), fontsize=7)

    plt.suptitle(
        f'Tuned PID  vs  PINN-MPC MPPI h=20  vs  PINN-MPC MPPI h=50  (v5)\n'
        f'Cessna 172  {INIT_ALT}ft -> {TARGET_ALT}ft  |  '
        f'PINN RMSE={best_rmse:.2f}ft',
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.savefig(f'{SAVE_DIR}/comparison_v5.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {SAVE_DIR}/comparison_v5.png')


plot_comparison(log_pid, log_mppi20, log_mppi50, TARGET_ALT)

## 12. Full Summary

In [ ]:
print('=== PINN HPO Progress ===')
for r in round_log:
    flag = '<< PASS' if r['rmse_ft'] <= RMSE_TARGET_FT else ''
    print(f"  Round {r['round']} | n={r['n_data']:,} | "
          f"RMSE={r['rmse_ft']:.3f}ft {flag}")
print(f'  Final RMSE  : {best_rmse:.3f}ft  (target {RMSE_TARGET_FT:.2f}ft)')
print(f'  Best params : {best_params}')

print('\n=== PID Optuna Result ===')
for k, v in bp.items():
    print(f'  {k:8s} = {v:.6f}')
print(f'  RMSE = {pid_study.best_value:.2f}ft')

print('\n=== Controller Comparison ===')
print(mdf.to_string())

print(f"""
=== Architecture Summary (v5 changes) ===
PINN:
  Activation  : SiLU (was Tanh)
  Depth       : 4 layers (was 3)
  Physics loss: structured kinematics + smoothness regulariser
  Rollout     : delta = d_norm * Y_std  (consistent, no Y_mean bias)

MPC:
  Algorithm   : MPPI (was CEM)
  Weighting   : soft exp(-cost/lam)  (was hard top-k)
  Horizon     : 20 / 50  (was 10 / 50)
  Cost        : altitude + pitch constraint penalty

Safety:
  Early termination on DIVERGE / CRASH / UNCONTROLLED

Paper points:
  1. MPPI vs CEM: smoother control, fewer constraint violations?
  2. Physics-consistent PINN: better rollout accuracy?
  3. Optuna PID: how much better than manual tuning?
  4. RMSE improvement per HPO round (iterative data collection)
""")

## 13. Save

In [ ]:
pd.DataFrame([bp]).to_csv(f'{SAVE_DIR}/pid_best_params.csv', index=False)
pd.DataFrame([{'round': r['round'],
               'n_data': r['n_data'],
               'rmse_ft': r['rmse_ft']}
              for r in round_log]
).to_csv(f'{SAVE_DIR}/hpo_log.csv', index=False)

print(f'Saved to {SAVE_DIR}/')
print('  pinn_best.pt          model + normalisation stats')
print('  pid_best_params.csv   Optuna PID gains')
print('  hpo_log.csv           PINN RMSE per round')
print('  metrics_v5.csv        controller comparison')
print('  comparison_v5.png     result plot')
print('  rollout_validation.png')
print()
print('Next steps:')
print('  v6: Uncertainty quantification (ensemble PINN)')
print('  v7: Multi-aircraft UAM environment')